In [1]:
%cd ..

c:\Users\HP\OneDrive - University of Moratuwa\Desktop\E-Vision-Projects\DB_SQL_GEN


In [2]:
from dotenv import load_dotenv

_ = load_dotenv()

In [3]:
from src.config import settings

In [4]:
# state.py
from typing import Annotated, Literal
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage

class AgentState(TypedDict):
    # ── auth (injected by FastAPI middleware, never modified by nodes) ──
    user_id: str
    role: Literal["rep", "asm", "rsm", "director"]
    allowed_rep_ids: list[str]
    allowed_regions: list[str]

    # ── conversation ──
    messages: Annotated[list[BaseMessage], add_messages]
    thread_id: str

    # ── routing ──
    intent: Literal["data_query", "kpi_query", "conversational", ""]
    kpi_names: list[str]           # extracted KPI names from user message

    # ── KPI resolution ──
    kpi_definitions: list[dict]    # fetched from vector store

    # ── SQL pipeline ──
    generated_sql: str
    validated_sql: str
    sql_retry_count: int
    query_results: list[dict]

    # ── analysis ──
    kpi_summary: dict
    threshold_breaches: list[str]

    # ── output ──
    action_suggestions: list[str]
    final_response: dict

In [5]:
# nodes/intent_router.py
import json
from langchain_anthropic import ChatAnthropic
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

# llm = ChatAnthropic(model="claude-sonnet-4-20250514", temperature=0)
llm = ChatOpenAI(name=settings.openai_model_fast, api_key=settings.openai_api_key)

SYSTEM_PROMPT = """You are an intent classifier for a sales KPI assistant.

Classify the user's latest message into exactly one of these intents:
  - "data_query"     → user wants actual data, numbers, a table (e.g. "show me my pipeline", "what were my deals last month")
  - "kpi_query"      → user is asking what a KPI means or how it is calculated (e.g. "what is win rate?", "how is quota attainment calculated?")
  - "conversational" → greeting, follow-up on a previous answer, small talk, or anything else

Also extract any KPI names mentioned (e.g. ["win_rate", "pipeline_coverage"]).

Respond ONLY with valid JSON, no markdown, no explanation:
{{
  "intent": "data_query" | "kpi_query" | "conversational",
  "kpi_names": ["snake_case_kpi_name", ...]
}}"""

async def intent_router_node(state: AgentState) -> dict:
    """
    Reads the latest user message from state.messages,
    calls LLM to classify intent, updates state.intent and state.kpi_names.
    """
    # Get the latest human message
    latest_message = next(
        (m for m in reversed(state["messages"]) if m.type == "human"),
        None
    )
    if not latest_message:
        return {"intent": "conversational", "kpi_names": []}

    response = await llm.ainvoke([
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=latest_message.content)
    ])

    try:
        parsed = json.loads(response.content)
        intent  = parsed.get("intent", "conversational")
        kpi_names = parsed.get("kpi_names", [])
    except (json.JSONDecodeError, AttributeError):
        intent = "conversational"
        kpi_names = []

    return {
        "intent": intent,
        "kpi_names": kpi_names,
    }

c:\Users\HP\OneDrive - University of Moratuwa\Desktop\E-Vision-Projects\DB_SQL_GEN\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
state = AgentState([{"role": "human", "content": "give me my productive call"}])

In [13]:
import nest_asyncio
from langchain_core.messages import HumanMessage

# Apply this once
nest_asyncio.apply()

# ====================== TEST ======================
async def test_intent():
    state = {
        "messages": [HumanMessage(content="what is my net sale")],
        "intent": "",
        "kpi_names": []
    }

    result = await intent_router_node(state)
    
    print("Intent:", result["intent"])
    print("KPI Names:", result["kpi_names"])


# Run it
await test_intent()

Intent: conversational
KPI Names: []


In [9]:
def route_intent(state: AgentState) -> str:
    """
    This is NOT a node. It is a pure function that reads state
    and returns a string key that LangGraph maps to the next node.

    add_conditional_edges("intent_router", route_intent, {
        "data_query":      "sql_generator",
        "kpi_query":       "kpi_lookup",
        "conversational":  "output_renderer",
    })
    """
    intent = state.get("intent", "conversational")

    if intent == "data_query":
        return "data_query"
    elif intent == "kpi_query":
        return "kpi_query"
    else:
        return "conversational"

In [13]:
# nodes/kpi_lookup.py
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="all-MiniLM-L6-v2")
vectorstore = Chroma(
    collection_name="business_definition",
    embedding_function=embeddings,
    persist_directory=settings.vector_store_path
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

async def kpi_lookup_node(state: AgentState) -> dict:
    """
    Uses state.kpi_names to fetch matching KPI definition docs
    from the vector store. Falls back to semantic search on the
    latest user message if no specific KPI names were extracted.
    """
    kpi_names = state.get("kpi_names", [])

    if kpi_names:
        query = " ".join(kpi_names)
    else:
        latest = next(
            (m.content for m in reversed(state["messages"]) if m.type == "human"),
            ""
        )
        query = latest

    docs = await retriever.ainvoke(query)

    kpi_definitions = [
        {
            "name": doc.metadata.get("kpi_name", "unknown"),
            "description": doc.page_content,
            "formula": doc.metadata.get("formula", ""),
            "thresholds": doc.metadata.get("thresholds", {}),
            "sql_hint": doc.metadata.get("sql_hint", ""),
        }
        for doc in docs
    ]

    return {"kpi_definitions": kpi_definitions}

In [24]:
# nodes/sql_generator.py
DB_SCHEMA = """
sales_flat
  Always include: Type  (distinguishes 'sales' vs 'return')
  For amounts:    in_sales
  For identity:   RepId (for joins), RepCode (for filtering/output),
                  RepName (if rep name needed in output)
  For customers:  CustomerCode (for joins), CustomerName (for output)
  For products:   ProductCode (for joins), ProductName (if needed)
  For time:       Date
  For invoices:   InvoiceNo
  For grouping:   ProductCategory, Brand, CustomerCategory, Route
  For territory:  ASM, RSM, Distributor  (only if territory filter needed)
  Optional:       Qty, Discount

sales_hierarchy_nodes
  Always include when this table is used: Id, Code
  Optional: Name  (only if RepName needed from this table)

sales_targets
  Always include when this table is used: RepId, Type, Value
  For date filtering: StartDate, EndDate
  Optional: Qty (only if target quantity needed),
            ProductId (only if product-level target),
            CustomerId (only if customer-level target)

external_parties
  For joins:    Id, Code
  For status:   Active (almost always needed to filter inactive customers)
  Optional:     Name (customer name from master — but prefer sf.CustomerName),
                Type (0=Customer, 1=Supplier, 2=Distributor),
                CreatedDate (only for new-customer queries)

products
  For joins:    Id, Code
  For volume:   Volume  (litres/kg — needed for ECO, CSD KPIs)
  Optional:     Name, PackSize

planned_routes
  Always include when used: PlannedDate, RouteId, RepId

route_customer_assignments
  Always include when used: RouteId, CustomerId
  
Relationships
──────────────
    sales_flat.RepId -> sales_hierarchy_nodes.Id, many-to-one,Sales records associated with each sales rep node
    sales_targets.RepId -> sales_hierarchy_nodes.Id, many-to-one, targets assigned for each sales rep node
    sales_flat.ProductCode -> Products.Code, many-to-one, Transaction details for each product
    sales_flat.CustomerCode -> external_parties.Code, many-to-one, Sales transactions associated with each customer
"""

def build_rbac_clause(state: AgentState) -> str:
    role = state["role"]
    if role == "rep":
        return f"rep_id = '{state['user_id']}'"
    elif role == "asm":
        ids = ", ".join(f"'{r}'" for r in state["allowed_rep_ids"])
        return f"rep_id IN ({ids})"
    elif role == "rsm":
        regions = ", ".join(f"'{r}'" for r in state["allowed_regions"])
        return f"region_id IN ({regions})"
    else:  # director — no filter
        return "1=1"

async def sql_generator_node(state: AgentState) -> dict:
    """
    Generates a read-only SQL query from the user's question.
    Injects RBAC context so the LLM includes the correct WHERE clause.
    Also injects any retrieved KPI definitions as SQL hints.
    """
    rbac_clause = build_rbac_clause(state)
    kpi_context = json.dumps(state.get("kpi_definitions", []), indent=2)
    print(kpi_context)
    latest_message = next(
        (m.content for m in reversed(state["messages"]) if m.type == "human"),
        ""
    )
    retry_count = state.get("sql_retry_count", 0)
    prev_sql = state.get("generated_sql", "")
    prev_error = state.get("sql_validation_error", "")

    retry_hint = ""
    if retry_count > 0 and prev_error:
        retry_hint = f"\n\nPrevious attempt failed validation:\nSQL: {prev_sql}\nError: {prev_error}\nFix the issue."

    system = f"""You generate safe, read-only PostgreSQL SELECT queries for a sales dashboard.

RBAC rule: ALWAYS include WHERE {rbac_clause} in every query.
Never use INSERT, UPDATE, DELETE, DROP, or CTEs that bypass the filter.

Database schema:
{DB_SCHEMA}

KPI definitions (use sql_hint as a guide):
{kpi_context}

Return ONLY the raw SQL query. No markdown, no explanation, no backticks.{retry_hint}"""

    response = await llm.ainvoke([
        SystemMessage(content=system),
        HumanMessage(content=latest_message)
    ])

    return {
        "generated_sql": response.content.strip(),
        "sql_retry_count": retry_count,
    }

In [25]:
# nodes/sql_validator.py
import sqlglot
import sqlglot.expressions as exp

ALLOWED_TABLES = {'sales_flat', "sales_targets", 'sales_hierarchy_nodes','external_parties', 
                                   'products', 'planned_routes', 'route_customer_assignments'}
FORBIDDEN_KEYWORDS = {"insert", "update", "delete", "drop", "truncate", "create", "alter"}
MAX_RETRIES = 3

def validate_sql(sql: str, rbac_clause_field: str) -> tuple[bool, str]:
    """Returns (is_valid, error_message)."""
    sql_lower = sql.lower()

    # Block DML/DDL by keyword scan
    for kw in FORBIDDEN_KEYWORDS:
        if kw in sql_lower:
            return False, f"Forbidden keyword '{kw}' detected."

    # Parse AST
    try:
        parsed = sqlglot.parse_one(sql, dialect="postgres")
    except sqlglot.errors.ParseError as e:
        return False, f"SQL parse error: {e}"

    # Only allow SELECT
    if not isinstance(parsed, exp.Select):
        return False, "Only SELECT statements are allowed."

    # Check referenced tables are in allowlist
    tables = {t.name.lower() for t in parsed.find_all(exp.Table)}
    bad_tables = tables - ALLOWED_TABLES
    if bad_tables:
        return False, f"Unauthorized tables referenced: {bad_tables}"

    # Check RBAC field is in the WHERE clause
    where_text = ""
    if parsed.find(exp.Where):
        where_text = parsed.find(exp.Where).sql().lower()
    if rbac_clause_field.lower() not in where_text and "1=1" not in where_text:
        return False, f"Missing RBAC filter: expected '{rbac_clause_field}' in WHERE clause."

    return True, ""


async def sql_validator_node(state: AgentState) -> dict:
    role = state["role"]
    if role == "rep":
        rbac_field = "rep_id"
    elif role == "asm":
        rbac_field = "rep_id"
    elif role == "rsm":
        rbac_field = "region_id"
    else:
        rbac_field = "1=1"

    is_valid, error = validate_sql(state["generated_sql"], rbac_field)

    if is_valid:
        return {
            "validated_sql": state["generated_sql"],
            "sql_validation_error": "",
        }
    else:
        return {
            "validated_sql": "",
            "sql_validation_error": error,
            "sql_retry_count": state.get("sql_retry_count", 0) + 1,
        }


def route_after_validation(state: AgentState) -> str:
    """Router function used with add_conditional_edges after sql_validator."""
    if state.get("validated_sql"):
        return "valid"
    if state.get("sql_retry_count", 0) >= MAX_RETRIES:
        return "failed"
    return "retry"

In [26]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

async def output_renderer_node(state: AgentState) -> dict:
    intent           = state.get("intent", "conversational")
    kpi_definitions  = state.get("kpi_definitions", [])
    query_results    = state.get("query_results", [])
    threshold_breaches = state.get("threshold_breaches", [])
    action_suggestions = state.get("action_suggestions", [])
    sql_validation_error = state.get("sql_validation_error", "")
    messages         = state.get("messages", [])

    # ── Build the parts of the response ──────────────────────────

    # PART 1: KPI definition card (only if we fetched one)
    definition_card = None
    if kpi_definitions:
        defn = kpi_definitions[0]
        definition_card = {
            "type":        "kpi_definition",
            "name":        defn["name"],
            "description": defn["description"],
            "formula":     defn.get("formula", ""),
            "thresholds":  defn.get("thresholds", {}),
        }

    # PART 2: Data table (only if query ran)
    data_table = None
    if query_results:
        data_table = {
            "type":    "data_table",
            "columns": list(query_results[0].keys()) if query_results else [],
            "rows":    query_results,
        }

    # PART 3: Natural language summary — LLM writes this
    context_parts = []
    if definition_card:
        context_parts.append(f"KPI definition: {json.dumps(definition_card)}")
    if data_table:
        context_parts.append(f"Query results: {json.dumps(query_results)}")
    if threshold_breaches:
        context_parts.append(f"Threshold breaches: {threshold_breaches}")
    if action_suggestions:
        context_parts.append(f"Suggested actions: {action_suggestions}")
    if sql_validation_error:
        context_parts.append(f"SQL error: {sql_validation_error}")

    latest_question = next(
        (m.content for m in reversed(messages) if m.type == "human"), ""
    )

    system = f"""You are a sales KPI assistant. The user asked: "{latest_question}"

You have been given structured data. Write a concise, helpful reply that:
1. Explains the KPI definition in plain language (1-2 sentences, if a definition was fetched)
2. States the user's current number clearly (if data was returned)
3. Compares it to the threshold and says whether it's healthy, at risk, or critical
4. If there are suggested actions, list them as bullet points under "Next steps"
5. If there was a SQL error, apologise and say you couldn't fetch the data

Context:
{chr(10).join(context_parts) if context_parts else "No data available."}

Role of the user: {state['role']}
Keep it concise. No markdown headers. Use plain sentences."""

    response = await llm.ainvoke([
        SystemMessage(content=system),
        HumanMessage(content=latest_question)
    ])

    summary_text = response.content

    # ── Assemble final response object ───────────────────────────
    final_response = {
        "definition": definition_card,  # → rendered as a definition card in UI
        "table":      data_table,       # → rendered as a data table in UI
        "summary":    summary_text,     # → rendered as chat reply
        "actions":    action_suggestions if action_suggestions else [],
    }

    # Append AI reply to conversation history
    return {
        "final_response": final_response,
        "messages": [AIMessage(content=summary_text)],
    }

In [27]:
# graph.py
from langgraph.graph import StateGraph, END
# from langgraph.checkpoint.redis import AsyncRedisCheckpointer

def build_graph():
    graph = StateGraph(AgentState)

    # ── Register nodes ──────────────────────────────────────────
    graph.add_node("intent_router",   intent_router_node)
    graph.add_node("kpi_lookup",      kpi_lookup_node)
    graph.add_node("sql_generator",   sql_generator_node)
    graph.add_node("sql_validator",   sql_validator_node)
    # graph.add_node("query_executor",  query_executor_node)
    # graph.add_node("kpi_analyser",    kpi_analyser_node)
    # graph.add_node("action_suggester",action_suggester_node)
    # graph.add_node("output_renderer", output_renderer_node)

    # ── Entry point ─────────────────────────────────────────────
    graph.set_entry_point("intent_router")

    # ── THE KEY LINE YOU ASKED ABOUT ────────────────────────────
    # After intent_router runs, call route_intent(state).
    # Its return value is matched against this dict to pick next node.
    graph.add_conditional_edges(
        "intent_router",      # source node
        route_intent,         # router function: reads state → returns string
        {
            "data_query":     "sql_generator",   # route_intent returned "data_query"
            "kpi_query":      "kpi_lookup",      # route_intent returned "kpi_query"
            # "conversational": "output_renderer", # route_intent returned "conversational"
        }
    )

    # kpi_lookup always feeds into sql_generator next
    # (we need definitions before generating SQL for KPI questions)
    graph.add_edge("kpi_lookup", "sql_generator")

    # SQL pipeline with retry loop
    graph.add_edge("sql_generator", "sql_validator")
    # graph.add_conditional_edges(
    #     "sql_validator",
    #     route_after_validation,
    #     {
    #         "valid":  "query_executor",   # SQL passed → run it
    #         "retry":  "sql_generator",    # SQL failed, under retry limit → regenerate
    #         "failed": "output_renderer",  # exceeded retries → return error message
    #     }
    # )

    # graph.add_edge("query_executor", "kpi_analyser")

    # # Only run action_suggester if there are threshold breaches
    # graph.add_conditional_edges(
    #     "kpi_analyser",
    #     route_after_analysis,
    #     {
    #         "breach": "action_suggester",
    #         "ok":     "output_renderer",
    #     }
    # )

    # graph.add_edge("action_suggester", "output_renderer")
    # graph.add_edge("output_renderer",  END)
    
    graph.add_edge("sql_validator",  END)

    # ── Compile with Redis checkpointer for multi-turn memory ───
    # checkpointer = AsyncRedisCheckpointer.from_conn_string("redis://localhost:6379")
    return graph.compile(
        # checkpointer=checkpointer
        )


compiled_graph = build_graph()

In [28]:
config = {"configurable": {"thread_id": "MATREP001"}}

initial_state = {
        # **auth_ctx,            # user_id, role, allowed_rep_ids, allowed_regions??
        "user_id": "MATREP001",
        "role": "rep",
        "allowed_rep_ids":["allowed_rep_ids"],
        "messages": [{"role": "human", "content": "what is my net sale"}],
        "intent": "",
        "kpi_names": [],
        "kpi_definitions": [],
        "generated_sql": "",
        "validated_sql": "",
        "sql_retry_count": 0,
        "query_results": [],
        "kpi_summary": {},
        "threshold_breaches": [],
        "action_suggestions": [],
        "final_response": {},
    }

result = await compiled_graph.ainvoke(initial_state, config=config)

[]


In [29]:
result

{'user_id': 'MATREP001',
 'role': 'rep',
 'allowed_rep_ids': ['allowed_rep_ids'],
 'messages': [HumanMessage(content='what is my net sale', additional_kwargs={}, response_metadata={}, id='f77ee30e-1884-4e4a-bfd5-b03f79ad5580')],
 'intent': 'data_query',
 'kpi_names': ['net_sales'],
 'kpi_definitions': [],
 'generated_sql': "SELECT SUM(in_sales) AS net_sale\nFROM sales_flat\nWHERE rep_id = 'MATREP001'\nAND Type = 'sales';",
 'validated_sql': "SELECT SUM(in_sales) AS net_sale\nFROM sales_flat\nWHERE rep_id = 'MATREP001'\nAND Type = 'sales';",
 'sql_retry_count': 0,
 'query_results': [],
 'kpi_summary': {},
 'threshold_breaches': [],
 'action_suggestions': [],
 'final_response': {}}

In [30]:
print(result['generated_sql'])

SELECT SUM(in_sales) AS net_sale
FROM sales_flat
WHERE rep_id = 'MATREP001'
AND Type = 'sales';
